In [ ]:
# Import first necessary libraries for data handling, statistics, and plotting.
# Each cell contains also individual libs to comment in if only specific part is needed.
# Some cells also contain optional print commands to check if you are actually getting the correct data and dimensions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Change file name here, if necessary
df = pd.read_csv('/content/AHOI_Survey_HumanAITeaming_FINISHED_TRUE_values.csv')

# Optional print cmds to check dimensions and first db entries
#print(f"Original DataFrame shape: {df.shape}")
#print("Original DataFrame head:")
#display(df.head())
print(df.columns)

**Correlation analysis between variables "age/sea experience" and the TIA questionnaire. The hypothesis is that older participants tend to be more skeptical, respectively, show less trust in automation processes (here after presentation of the maritime assistant).**

In [ ]:
# Remove the first two rows, which contain metadata
df_cleaned = df.iloc[2:].copy()

# Convert 'age' and 'sea_service' columns to numeric type
df_cleaned['age'] = pd.to_numeric(df_cleaned['age'], errors='coerce')
df_cleaned['sea_service'] = pd.to_numeric(df_cleaned['sea_service'], errors='coerce')

print("DataFrame after removing metadata rows and converting 'age' and 'sea_service' to numeric:")
display(df_cleaned[['age', 'sea_service']].head())
print(f"Shape of the cleaned DataFrame: {df_cleaned.shape}")

Optional print commands to check data.

In [ ]:
#print("Unique values in 'age' column:")
#print(df_cleaned['age'].unique())

#print("\nUnique values in 'sea_service' column:")
#print(df_cleaned['sea_service'].unique())

In [ ]:
if 'df_cleaned' not in globals():
    print("Error: DataFrame 'df_cleaned' is not defined. Please ensure that the data loading and cleaning steps (cells `cvVUzflgDGI1` and `b8b944b0`) have been executed successfully before running this cell.")
    raise NameError("DataFrame 'df_cleaned' is not defined. Run preceding cells.")

# Get TIA data for Scenario 1 and Scenario 2
tia_scenario1_cols = [col for col in df_cleaned.columns if 'Scenario1_TIA_' in col]
tia_scenario2_cols = [col for col in df_cleaned.columns if 'Scenario2_TIA_' in col]

# Convert TIA columns to numeric (coercing errors to NaN)
df_cleaned[tia_scenario1_cols] = df_cleaned[tia_scenario1_cols].apply(pd.to_numeric, errors='coerce')
df_cleaned[tia_scenario2_cols] = df_cleaned[tia_scenario2_cols].apply(pd.to_numeric, errors='coerce')

# Define items to be reverse-coded
reverse_code_items = ['_6', '_7', '_8', '_9', '_10', '_11']

# Apply reverse-coding to specified TIA items (assuming a 5-point scale: new_value = 6 - old_value)
for col_suffix in reverse_code_items:
    col_s1 = f'Scenario1_TIA{col_suffix}'
    col_s2 = f'Scenario2_TIA{col_suffix}'
    if col_s1 in df_cleaned.columns:
        df_cleaned[col_s1] = 6 - df_cleaned[col_s1]
    if col_s2 in df_cleaned.columns:
        df_cleaned[col_s2] = 6 - df_cleaned[col_s2]

# Calculate average TIA score for each scenario AFTER reverse-coding
df_cleaned['Scenario1_TIA_score'] = df_cleaned[tia_scenario1_cols].mean(axis=1)
df_cleaned['Scenario2_TIA_score'] = df_cleaned[tia_scenario2_cols].mean(axis=1)

print("Calculated Scenario1_TIA_score and Scenario2_TIA_score (with reverse-coding). Head of relevant columns:")
display(df_cleaned[['age', 'sea_service', 'Scenario1_TIA_score', 'Scenario2_TIA_score']].head())

# Calculate Spearman correlations
correlation_matrix = df_cleaned[['age', 'sea_service', 'Scenario1_TIA_score', 'Scenario2_TIA_score']].corr(method='spearman')
display(correlation_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Spearman Correlation Matrix of Age, Sea Service, and TIA Scores (Reverse-Coded)')
plt.show()

### Distribution of TIA Scores by Age Group (Box Plots)

In [ ]:
age_labels = {1.0: '18-24', 2.0: '25-34', 3.0: '35-44', 4.0: '45-54', 5.0: '>54'}
df_cleaned['age_labels'] = df_cleaned['age'].map(age_labels)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.boxplot(x='age_labels', y='Scenario1_TIA_score', data=df_cleaned, palette='viridis', hue='age_labels', legend=False)
plt.title('Scenario 1 TIA Score Distribution by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Scenario 1 TIA Score')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.boxplot(x='age_labels', y='Scenario2_TIA_score', data=df_cleaned, palette='magma', hue='age_labels', legend=False)
plt.title('Scenario 2 TIA Score Distribution by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Scenario 2 TIA Score')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Kruskal-Wallis H-test for TIA Scores Across Age Groups

To formally test if there are statistically significant differences in TIA scores across different age groups, we will use the Kruskal-Wallis H-test (non-parametric test).

In [ ]:
from scipy.stats import kruskal

# Drop rows where 'age' is NaN to ensure valid age groups for the test
df_kruskal = df_cleaned.dropna(subset=['age']).copy()

# Get unique age groups
age_groups = sorted(df_kruskal['age'].unique())

print(f"Age Groups found: {age_groups}")

# Prepare data for Kruskal-Wallis test for Scenario 1 TIA scores
tia1_scores_by_age = [df_kruskal[df_kruskal['age'] == group]['Scenario1_TIA_score'].dropna() for group in age_groups]

# Perform Kruskal-Wallis H-test for Scenario 1
H_stat_s1, p_value_s1 = kruskal(*tia1_scores_by_age)

print(f"\nKruskal-Wallis H-test for Scenario 1 TIA Scores:")
print(f"  H-statistic = {H_stat_s1:.3f}")
print(f"  p-value = {p_value_s1:.3f}")

# Prepare data for Kruskal-Wallis test for Scenario 2 TIA scores
tia2_scores_by_age = [df_kruskal[df_kruskal['age'] == group]['Scenario2_TIA_score'].dropna() for group in age_groups]

# Perform Kruskal-Wallis H-test for Scenario 2
H_stat_s2, p_value_s2 = kruskal(*tia2_scores_by_age)

print(f"\nKruskal-Wallis H-test for Scenario 2 TIA Scores:")
print(f"  H-statistic = {H_stat_s2:.3f}")
print(f"  p-value = {p_value_s2:.3f}")

if p_value_s1 < 0.05 or p_value_s2 < 0.05:
    print("\nInterpretation: At least one scenario shows a statistically significant difference in TIA scores between age groups.")
    print("  Further post-hoc tests (e.g., Mann-Whitney U with correction) would be needed to identify which specific age groups differ.")
else:
    print("\nInterpretation: No statistically significant difference in TIA scores found between age groups for either scenario.")

### Descriptive Statistics for TIA Scores by Age Group

In [ ]:
print("Descriptive statistics for Scenario 1 TIA scores by age group:")
display(df_cleaned.groupby('age')['Scenario1_TIA_score'].describe())

print("\nDescriptive statistics for Scenario 2 TIA scores by age group:")
display(df_cleaned.groupby('age')['Scenario2_TIA_score'].describe())

### Mean TIA Scores by Age Group (Bar Charts)

In [ ]:
age_labels = {1.0: '18-24', 2.0: '25-34', 3.0: '35-44', 4.0: '45-54', 5.0: '>54'}

# Calculate mean TIA scores by age group
mean_tia_scores_by_age = df_cleaned.groupby('age')[['Scenario1_TIA_score', 'Scenario2_TIA_score']].mean().reset_index()
mean_tia_scores_by_age['age_labels'] = mean_tia_scores_by_age['age'].map(age_labels)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.barplot(x='age_labels', y='Scenario1_TIA_score', data=mean_tia_scores_by_age, palette='viridis', hue='age_labels', legend=False)
plt.title('Mean Scenario 1 TIA Score by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Mean Scenario 1 TIA Score')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.barplot(x='age_labels', y='Scenario2_TIA_score', data=mean_tia_scores_by_age, palette='magma', hue='age_labels', legend=False)
plt.title('Mean Scenario 2 TIA Score by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Mean Scenario 2 TIA Score')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Number of Participants per Age Group

In [ ]:
import matplotlib.ticker as plticker

age_labels = {1.0: '18-24', 2.0: '25-34', 3.0: '35-44', 4.0: '45-54', 5.0: '>54'}

# Count participants in each age group
participant_counts = df_cleaned['age'].value_counts().sort_index().reset_index()
participant_counts.columns = ['Age Group', 'Number of Participants']
participant_counts['Age Group Labels'] = participant_counts['Age Group'].map(age_labels)

plt.figure(figsize=(8, 6))
sns.barplot(x='Age Group Labels', y='Number of Participants', data=participant_counts, color='dimgray')
plt.title('Number of Participants per Age Group')
plt.xlabel('Years')
plt.ylabel('Number of Participants')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_locator(plticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

**Correlation analysis between variables "age/sea experience" and sentiment analysis of advantages/disadvantages of using a maritime assistant. The hypothesis is that older participants tend to express more negative sentiment than younger participants.**

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

age_labels = {1.0: '18-24', 2.0: '25-34', 3.0: '35-44', 4.0: '45-54', 5.0: '>54'}

# Download necessary NLTK data for sentiment analysis
nltk.download('vader_lexicon')
nltk.download('wordnet')
nltk.download('punkt')

# Ensure 'advantages' and 'disadvantages' columns are string type and fill NaN with empty strings
df_cleaned['advantages'] = df_cleaned['advantages'].astype(str).fillna('')
df_cleaned['disadvantages'] = df_cleaned['disadvantages'].astype(str).fillna('')

# Function to get sentiment polarity using TextBlob
def get_sentiment_polarity(text):
    return TextBlob(text).sentiment.polarity

# Apply sentiment analysis to 'advantages' and 'disadvantages'
df_cleaned['advantages_sentiment'] = df_cleaned['advantages'].apply(get_sentiment_polarity)
df_cleaned['disadvantages_sentiment'] = df_cleaned['disadvantages'].apply(get_sentiment_polarity)

# Group by 'age' and calculate the mean sentiment scores
sentiment_by_age = df_cleaned.groupby('age')[['advantages_sentiment', 'disadvantages_sentiment']].mean().reset_index()
sentiment_by_age['age_labels'] = sentiment_by_age['age'].map(age_labels)

print("Average sentiment polarity for 'advantages' and 'disadvantages' by age group:")
display(sentiment_by_age)

# Visualize the sentiment by age group
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.barplot(x='age_labels', y='advantages_sentiment', data=sentiment_by_age, palette='viridis')
plt.title('Average Sentiment of "Advantages" by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Sentiment Polarity')

plt.subplot(1, 2, 2)
sns.barplot(x='age_labels', y='disadvantages_sentiment', data=sentiment_by_age, palette='magma')
plt.title('Average Sentiment of "Disadvantages" by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Sentiment Polarity')

plt.tight_layout()
plt.show()

# Correlation age/sea experience with ATAS (pre-questionnaire)

In [ ]:
# Get ATAS data
atas_cols = [col for col in df_cleaned.columns if 'ATAS_' in col and col != 'ATAS_score']

# Convert ATAS columns to numeric (coercing errors to NaN)
df_cleaned[atas_cols] = df_cleaned[atas_cols].apply(pd.to_numeric, errors='coerce')

# Calculate average ATAS score
df_cleaned['ATAS_score'] = df_cleaned[atas_cols].mean(axis=1)

print("Calculated ATAS_score. Head of relevant columns:")
#display(df_cleaned[['age', 'sea_service', 'ATAS_score']].head())

# Calculate Spearman correlations including ATAS_score
correlation_matrix_atas = df_cleaned[['age', 'sea_service', 'ATAS_score']].corr(method='spearman')

print("\nCorrelation Matrix (including ATAS Scores) using Spearman's method:")
#display(correlation_matrix_atas)

plt.figure(figsize=(6, 5))
sns.heatmap(correlation_matrix_atas, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Spearman Correlation Matrix of Age, Sea Service, and ATAS Scores')
plt.show()

### Kruskal-Wallis H-test for ATAS Scores Across Age Groups and Sea Service Groups

We will use the Kruskal-Wallis H-test to formally assess if there are statistically significant differences in ATAS scores (pre-survey questionnaire) based on age groups and sea service groups. This non-parametric test is suitable for comparing three or more independent groups.

In [ ]:
from scipy.stats import kruskal

# Data prep
df_kruskal_age_atas = df_cleaned.dropna(subset=['age', 'ATAS_score']).copy()
age_groups_atas = sorted(df_kruskal_age_atas['age'].unique())

print(f"Age Groups found for ATAS test: {age_groups_atas}")

# Prepare data for Kruskal-Wallis test for ATAS scores by age
atas_scores_by_age = [df_kruskal_age_atas[df_kruskal_age_atas['age'] == group]['ATAS_score'].dropna() for group in age_groups_atas]

# Perform test (ATAS vs. age group)
H_stat_atas_age, p_value_atas_age = kruskal(*atas_scores_by_age)

print(f"\nKruskal-Wallis H-test for ATAS Scores by Age Group:")
print(f"  H-statistic = {H_stat_atas_age:.3f}")
print(f"  p-value = {p_value_atas_age:.3f}")

if p_value_atas_age < 0.05:
    print("  Interpretation: There is a statistically significant difference in ATAS scores between age groups.")
else:
    print("  Interpretation: No statistically significant difference in ATAS scores found between age groups.")


# Data prep
df_kruskal_sea_atas = df_cleaned.dropna(subset=['sea_service', 'ATAS_score']).copy()
sea_service_groups_atas = sorted(df_kruskal_sea_atas['sea_service'].unique())

print(f"\nSea Service Groups found for ATAS test: {sea_service_groups_atas}")

# Prepare data for Kruskal-Wallis test for ATAS scores by sea service
atas_scores_by_sea = [df_kruskal_sea_atas[df_kruskal_sea_atas['sea_service'] == group]['ATAS_score'].dropna() for group in sea_service_groups_atas]

# Perform test
H_stat_atas_sea, p_value_atas_sea = kruskal(*atas_scores_by_sea)

print(f"\nKruskal-Wallis H-test for ATAS Scores by Sea Service Group:")
print(f"  H-statistic = {H_stat_atas_sea:.3f}")
print(f"  p-value = {p_value_atas_sea:.3f}")

if p_value_atas_sea < 0.05:
    print("  Interpretation: There is a statistically significant difference in ATAS scores between sea service groups.")
else:
    print("  Interpretation: No statistically significant difference in ATAS scores found between sea service groups.")

### Kruskal-Wallis H-test for TIA Scores Across Sea Service Groups

We will use the Kruskal-Wallis H-test (non-parametric test) to formally assess if there are statistically significant differences in TIA scores based on sea service groups.

In [ ]:
from scipy.stats import kruskal

# Data prep
df_kruskal_sea_tia = df_cleaned.dropna(subset=['sea_service']).copy()
sea_service_groups = sorted(df_kruskal_sea_tia['sea_service'].unique())

#print(f"Sea Service Groups found: {sea_service_groups}")

# Prepare data to perform Kruskal-Wallis test Scenario 1
tia1_scores_by_sea = [df_kruskal_sea_tia[df_kruskal_sea_tia['sea_service'] == group]['Scenario1_TIA_score'].dropna() for group in sea_service_groups]
# test
H_stat_s1_sea, p_value_s1_sea = kruskal(*tia1_scores_by_sea)

print(f"\nKruskal-Wallis H-test for Scenario 1 TIA Scores by Sea Service Group:")
print(f"  H-statistic = {H_stat_s1_sea:.3f}")
print(f"  p-value = {p_value_s1_sea:.3f}")

# Prepare data to perform Kruskal-Wallis test Scenario 2
tia2_scores_by_sea = [df_kruskal_sea_tia[df_kruskal_sea_tia['sea_service'] == group]['Scenario2_TIA_score'].dropna() for group in sea_service_groups]
# test
H_stat_s2_sea, p_value_s2_sea = kruskal(*tia2_scores_by_sea)

print(f"\nKruskal-Wallis H-test for Scenario 2 TIA Scores by Sea Service Group:")
print(f"  H-statistic = {H_stat_s2_sea:.3f}")
print(f"  p-value = {p_value_s2_sea:.3f}")

if p_value_s1_sea < 0.05 or p_value_s2_sea < 0.05:
    print("\nInterpretation: At least one scenario shows a statistically significant difference in TIA scores between sea service groups.")
    print("  Further post-hoc tests (e.g., Mann-Whitney U with correction) would be needed to identify which specific sea service groups differ.")
else:
    print("\nInterpretation: No statistically significant difference in TIA scores found between sea service groups for either scenario.")

### Variance of TIA Scores by Age Group

Calculate the variance of Scenario 1 and Scenario 2 TIA scores within each age group. This will show the spread of scores within each group, which contributes to the overall variability.

In [ ]:
# Calculate variance of Scenario 1 TIA scores by age group
print("Variance of Scenario 1 TIA Scores by Age Group:")
display(df_cleaned.groupby('age')['Scenario1_TIA_score'].var().reset_index())

# Calculate variance of Scenario 2 TIA scores by age group
print("\nVariance of Scenario 2 TIA Scores by Age Group:")
display(df_cleaned.groupby('age')['Scenario2_TIA_score'].var().reset_index())

### Post-Hoc Analysis for Scenario 2 TIA Scores by Age Group (Dunn's Test)


In [ ]:
# Install scikit_posthocs if not already installed
import sys
!{sys.executable} -m pip install scikit_posthocs

import scikit_posthocs as sp

In [ ]:
df_posthoc_s2 = df_cleaned.dropna(subset=['age', 'Scenario2_TIA_score']).copy()

# 'age' is treated as a categorical variable for grouping in post-hoc tests
df_posthoc_s2['age'] = df_posthoc_s2['age'].astype('category')

#print("Performing Dunn's test for Scenario 2 TIA Scores across age groups...")

# Perform Dunn's post-hoc test
# The 'group_by' argument specifies the column used for grouping
# The 'val_col' argument specifies the column containing the values to be compared
# The 'p_adjust' argument specifies the method for p-value adjustment (here: 'bonferroni')
dunn_results_s2 = sp.posthoc_dunn(df_posthoc_s2, val_col='Scenario2_TIA_score', group_col='age', p_adjust='bonferroni')

print("\nDunn's Post-Hoc Test Results for Scenario 2 TIA Scores by Age Group (Bonferroni-adjusted p-values):")
#display(dunn_results_s2)

# Optional: Visualize the p-value matrix
plt.figure(figsize=(8, 6))
sns.heatmap(dunn_results_s2, annot=True, cmap='viridis', fmt=".3f", linewidths=.5)
plt.title('Dunn\'s Test p-values for Scenario 2 TIA Scores by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Age Group')
plt.show()